### Day 13: LangGraph Patterns — Loops, Branches & Supervisor 🎯

Today you master production-grade LangGraph patterns — retry logic, fallback branches, streaming state, and the supervisor pattern that orchestrates multiple specialized agents. This is the direct foundation of Project 3.

##### 1. What You're Learning Today
- Pattern 1: Retry Loop        → retry failed operations automatically
- Pattern 2: Fallback Branch   → graceful degradation on failure  
- Pattern 3: Map-Reduce        → process many items in parallel
- Pattern 4: Supervisor        → one agent coordinates many agents
- Pattern 5: State Streaming   → watch graph execute in real time

In [1]:
# Cell 1: Imports
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, List, Optional, Literal
import operator, json, time, os
from duckduckgo_search import DDGS

load_dotenv()
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("✅ Ready")

d:\AI\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Ready


##### 2. Pattern 1 — Retry Loop with Error Handling

In [2]:
# Cell 2: Retry logic - critical for production agents

class RetryState(TypedDict):
    messages:     Annotated[List, add_messages]
    task:         str
    result:       Optional[str]
    attempts:     int
    max_attempts: int
    last_error:   Optional[str]
    success:      bool


def attempt_task_node(state: RetryState) -> dict:
    """
    Try to complete the task.
    Simulates occasional failures for demo.
    """
    attempt = state["attempts"] + 1
    print(f"attempt_task (try {attempt}/{state['max_attempts']})")

    try:
        response = llm.invoke([
            SystemMessage(content="""Complete the task.
Respond with valid JSON only:
{"result": "your answer", "confidence": 0.0-1.0}"""),
            HumanMessage(content=f"Task: {state['task']}")
        ])

        # Parse JSON response
        data = json.loads(response.content.strip())

        # Simulate failure if confidence is low
        if data.get("confidence", 0) < 0.6 and attempt < state["max_attempts"]:
            raise ValueError(
                f"Low confidence: {data.get('confidence')}"
            )

        return {
            "result":    data.get("result", response.content),
            "attempts":  attempt,
            "success":   True,
            "last_error": None
        }

    except json.JSONDecodeError as e:
        return {
            "attempts":   attempt,
            "success":    False,
            "last_error": f"JSON parse error: {e}"
        }
    except Exception as e:
        return {
            "attempts":   attempt,
            "success":    False,
            "last_error": str(e)
        }


def handle_failure_node(state: RetryState) -> dict:
    """Called when all retries exhausted"""
    print("handle_failure")
    return {
        "result": f"Task failed after {state['attempts']} attempts. "
                  f"Last error: {state['last_error']}"
    }


def should_retry(state: RetryState) -> str:
    """Router: retry, succeed, or give up"""
    if state["success"]:
        print(f"     → Success on attempt {state['attempts']}")
        return "success"

    if state["attempts"] >= state["max_attempts"]:
        print(f"     → Max attempts reached, giving up")
        return "give_up"

    wait = state["attempts"] * 1  # exponential-ish backoff
    print(f"     → Retrying in {wait}s (error: {state['last_error']})")
    time.sleep(wait)
    return "retry"


# Build retry graph
retry_graph = StateGraph(RetryState)
retry_graph.add_node("attempt_task",   attempt_task_node)
retry_graph.add_node("handle_failure", handle_failure_node)

retry_graph.add_edge(START, "attempt_task")
retry_graph.add_conditional_edges(
    "attempt_task",
    should_retry,
    {
        "retry":    "attempt_task",     # ← loop back
        "success":  END,
        "give_up":  "handle_failure"
    }
)
retry_graph.add_edge("handle_failure", END)

retry_runner = retry_graph.compile()

print("Testing retry pattern:\n")
result = retry_runner.invoke({
    "task":         "Explain what LangGraph is in one sentence",
    "attempts":     0,
    "max_attempts": 3,
    "success":      False,
    "messages":     []
})
print(f"\n✅ Result: {result['result']}")
print(f"   Attempts: {result['attempts']}")
print(f"   Success:  {result['success']}")

Testing retry pattern:

attempt_task (try 1/3)
     → Success on attempt 1

✅ Result: LangGraph is an artificial intelligence model developed by Meta, designed to process and generate human-like language, similar to other large language models.
   Attempts: 1
   Success:  True


##### Pattern 2 — Fallback Branch